# Lab 3 · Lesson 1: BeautifulSoup in depth
Lab 1 taught you CSS selectors and XPath. Today: everything **around** the selector, on a real shop (LDLC, graphics
cards). Choosing a parser, moving in the tree, cleaning text, tables with `rowspan`, going faster ... and the most
important habit: **search the page source for the value before you think "I need a browser"**.

In [82]:
import json
import re
import time
from io import StringIO
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup, SoupStrainer

USER_AGENT = "Mozilla/5.0 (compatible; ESILV-Lab-Advanced/1.0; educational use)"
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "fr-FR,fr;q=0.9"})

LISTING_URL = "https://www.ldlc.com/informatique/pieces-informatique/carte-graphique-interne/c4684/"
PRODUCT_URL = "https://www.ldlc.com/fiche/PB00686694.html"

# two pages, downloaded ONCE: every cell below works on these variables (no new request)
listing_html = session.get(LISTING_URL, timeout=20).text
time.sleep(2)

product_html = session.get(PRODUCT_URL, timeout=20).text
print(f"listing: {len(listing_html) // 1024} KB, product: {len(product_html) // 1024} KB")

listing: 460 KB, product: 180 KB


## 1. Choose a parser
BeautifulSoup does not read HTML itself: it asks a **parser**. Two are installed:

| Parser | Speed | Broken HTML | Use it |
|---|---|---|---|
| `"lxml"` | fast (C) | repairs it well | **by default** |
| `"html.parser"` | slower (pure Python) | repairs it differently | when lxml is not installed |

The same broken HTML can give **different trees**. Always write the parser name (no warning, same result everywhere).

In the output below, compare the two trees: `lxml` closes each `<li>` and `<p>` where a browser would, but
`html.parser` puts the second `<li>` **inside** the first one and the second `<p>` **inside** `<b>`.

In [83]:
# broken HTML, written like a real page so you can see what is MISSING:
#   both <li> are never closed (no </li>)
#   <b> is never closed (no </b>)
#   both <p> are never closed (no </p>)
broken = """
<ul>
  <li>one
  <li>two
</ul>
<p>a <b>bold
<p>new paragraph
"""

for parser in ("lxml", "html.parser"):
    start = time.perf_counter()
    for _ in range(3):
        BeautifulSoup(listing_html, parser)
    ms = (time.perf_counter() - start) / 3 * 1000

    print(f"===== {parser}: {ms:.0f} ms per parse of the listing page =====")
    print(BeautifulSoup(broken, parser).prettify())

===== lxml: 132 ms per parse of the listing page =====
<html>
 <body>
  <ul>
   <li>
    one
   </li>
   <li>
    two
   </li>
  </ul>
  <p>
   a
   <b>
    bold
   </b>
  </p>
  <p>
   new paragraph
  </p>
 </body>
</html>

===== html.parser: 150 ms per parse of the listing page =====
<ul>
 <li>
  one
  <li>
   two
  </li>
 </li>
</ul>
<p>
 a
 <b>
  bold
  <p>
   new paragraph
  </p>
 </b>
</p>



## 2. Find things: `select` first
`select()` / `select_one()` take a CSS selector (lab 1). `find()` / `find_all()` take a tag name + attributes.
Both work; CSS is shorter and you can test it in DevTools first (`Ctrl+F` in the Elements tab).

| You want | CSS selector |
|---|---|
| an attribute exists | `li[data-id]` |
| an attribute starts with / contains | `a[href^="/fiche/"]`, `[class*="star-"]` |
| a direct child | `ul.pagination > li` |
| the n-th of its type | `tr:nth-of-type(2)` |
| an element that contains a text | `td:-soup-contains("Mémoire")` (BeautifulSoup only) |

In [84]:
soup = BeautifulSoup(listing_html, "lxml")

# every product card on the listing page
cards = soup.select("li.pdt-item")
total_products = len(cards)
print("total products on this page:", total_products)

total products on this page: 48


In [85]:
# Method 1: one card only (the first one): its title link gives the product name and the product URL
first = cards[0]
title_link = first.select_one("h3 a")
product_name = title_link.get_text(strip=True)
product_href = title_link["href"]
print("first product name:", product_name)
print("first product link:", product_href)

# Method 2: card by card: loop over every card and read its title link (name + URL together)
# for card in cards:
#     title_link = card.select_one("h3 a")
#     product_name = title_link.get_text(strip=True)
#     product_href = title_link["href"]
#     print(product_name, "->", product_href)

# Method 3 (shortcut): only the URLs, one selector for every link that points to a product page
product_links = [a["href"] for a in soup.select('a[href^="/fiche/"]')]
print("first 3 product links:", product_links[:3])

# the current page number in the pagination bar at the bottom
current_page = soup.select_one("ul.pagination > li.active").get_text(" ", strip=True)
print("current page:", current_page)

first product name: ASRock AMD Radeon RX 9060 XT Challenger 16GB OC
first product link: /fiche/PB00686694.html
first 3 product links: ['/fiche/PB00671029.html', '/fiche/PB00748328.html', '/fiche/PB00671174.html']
current page: 1 (current)


## 3. Move in the tree
Sometimes there is no good selector for the value, but there is one for a **label next to it**.
Find the label, then move: `find_next_sibling()` (right), `find_previous_sibling()` (left), `find_parent()` (up).

In [86]:
product = BeautifulSoup(product_html, "lxml")

# step 1: find the LABEL cell (its text is stable, so it is easy to select)
label_cell = product.select_one('#product-parameters td:-soup-contains("Taille mémoire vidéo")')
label_text = label_cell.get_text(strip=True)
print("label:", label_text)

# step 2: move RIGHT to the next <td> on the same row: that is the VALUE
value_cell = label_cell.find_next_sibling("td")
value_text = " ".join(value_cell.get_text(" ").split())
print("value:", value_text)

label: Taille mémoire vidéo
value: 16 Go


## 4. Text: `.string`, `.get_text()`, and cleaning a price
- `.string` is `None` as soon as the tag has more than one child. Prefer `.get_text()`.
- `get_text(" ", strip=True)`: a space between the pieces, no spaces around them.
- A price on the page is **text** (`"969€95"`). To sort or compare prices, turn it into a **number** (`969.95`).
- French sites write big numbers with **non-breaking spaces** (`"1\xa0129€95"`). They look like spaces but are not
  `" "`. `re.sub(r"\s", "", text)` removes all of them.

In [87]:
price_tag = product.select_one(".product-infoPrice .price")
print("the tag:", price_tag)                          # 969€<sup>95</sup>: the cents are in a <sup>

price_string = price_tag.string                      # None: the tag has more than one child (text + <sup>)
price_raw = price_tag.get_text()                     # the pieces glued together
price_clean = price_tag.get_text(" ", strip=True)    # a space between the pieces
print("string   :", repr(price_string))
print("get_text :", repr(price_raw))
print("clean    :", repr(price_clean))

# from text to a NUMBER you can sort and compare
price_no_spaces = re.sub(r"\s", "", price_raw)       # removes every space, invisible ones too ("1\xa0129€95" -> "1129€95")
price_number = float(price_no_spaces.replace("€", "."))   # "969€95" -> "969.95" -> 969.95
print("number   :", price_number)

the tag: <div class="price"><div class="price">969€<sup>95</sup></div></div>
string   : None
get_text : '969€95'
clean    : '969€ 95'
number   : 969.95


## 5. Attributes: the rating is hidden in a class name
On the listing page, each card shows stars and "5 avis". The stars are **pictures** (CSS): there is no "5/5" text
to read with `get_text()`. Right-click the stars of the first product → Inspect:

```html
<div class="ratingClient">
  <a href="/fiche/PB00686694.html#clients-review">
    <span class="star-10"></span> 5 avis
  </a>
</div>
```

The rating is in the **attribute** `class="star-10"`: 10 half-stars = **5 / 5**.
Read an attribute with `tag["class"]`. Careful: `class` is a **list** (a tag can have several classes).

In [88]:
# step 1: select the stars <span> inside the rating block of the first card
stars_tag = first.select_one(".ratingClient span[class^='star-']")

# step 2: read its class attribute (a list) and take the first class
stars_class = stars_tag["class"][0]          # "star-10"

# step 3: turn it into a rating out of 5
half_stars = int(stars_class.split("-")[1])  # "star-10" -> 10
rating = half_stars / 2                      # 10 half-stars -> 5.0
print("class:", stars_class, "-> rating:", rating, "/ 5")

class: star-10 -> rating: 5.0 / 5


## 6. Relative links
`href="/fiche/PB00686694.html"` is relative. `urljoin(page_url, href)` builds the absolute URL (it also handles
`../x`, `?page=2`, and links that are already absolute).

In [89]:
product_href = first.select_one("h3 a")["href"]      # relative: "/fiche/..."
example_links = [product_href, "page2/", "?sort=price", "https://media.ldlc.com/x.jpg"]

for link in example_links:
    absolute_url = urljoin(LISTING_URL, link)
    print(f"{link:28} -> {absolute_url}")

/fiche/PB00686694.html       -> https://www.ldlc.com/fiche/PB00686694.html
page2/                       -> https://www.ldlc.com/informatique/pieces-informatique/carte-graphique-interne/c4684/page2/
?sort=price                  -> https://www.ldlc.com/informatique/pieces-informatique/carte-graphique-interne/c4684/?sort=price
https://media.ldlc.com/x.jpg -> https://media.ldlc.com/x.jpg


## 7. Tables: `pandas.read_html`, and when it is not enough
`pd.read_html` turns every `<table>` into a DataFrame in one line. It understands `rowspan` ... but it **repeats**
the merged cell on every row, and it does not know which cells are noise (hidden labels).

In [90]:
parameters_table = product.select_one("#product-parameters")
tables = pd.read_html(StringIO(str(parameters_table)))   # one DataFrame per <table>
sheet = tables[0]

rows, columns = sheet.shape
print("rows:", rows, "| columns:", columns)
sheet.head(8)

rows: 39 | columns: 3


,0,1,2
0,Informations générales,Désignation,ASRock AMD Radeon RX 9060 XT Challenger 16GB OC
1,Informations générales,Marque,ASRock ASRock
2,Informations générales,Modèle,RX9060XT CL 16GO
3,Chipset graphique,Chipset graphique,AMD Radeon RX 9060 XT
4,Chipset graphique,GPU series,AMD Radeon RX Série 9000
5,Chipset graphique,Marque du GPU,AMD
6,Chipset graphique,Fréquence boostée,3290 MHz
7,Chipset graphique,Overclockée,Oui


## 8. Change the tree: `extract()` and `decompose()`
Sometimes `get_text()` gives you **more** than what you see on screen: the HTML contains hidden text (labels,
text for screen readers, `<script>`, ads). Do not clean the string afterwards: **remove the noise tag from the tree
first**, then call `get_text()`.
- `tag.extract()` removes the tag and gives it back (you can still print or use it);
- `tag.decompose()` removes the tag and destroys it (for noise you never want back).

In [91]:
# Example 1, on the PRODUCT page: the row "Marque" (brand) in the "Fiche technique" table (sections 3 and 7).
# On screen you see "ASRock" once. In the HTML the value cell has the name TWICE:
#   <td class="checkbox">
#     <input type="checkbox" ...> <label ...>ASRock</label>   <- hidden filter checkbox (not shown on screen)
#     <a href="...">ASRock</a>                               <- the link you see
#   </td>
brand_cell = product.select_one('#product-parameters td:-soup-contains("Marque") + td')
brand_before = " ".join(brand_cell.get_text(" ").split())
print("before:", brand_before)                                # "ASRock ASRock"

# remove the hidden <label> tags from the tree; decompose() destroys them
hidden_labels = brand_cell.find_all("label")
for hidden in hidden_labels:
    hidden.decompose()

brand_after = " ".join(brand_cell.get_text(" ").split())
print("after :", brand_after)                                 # "ASRock"

before: ASRock ASRock
after : ASRock


In [92]:
# Example 2, on the LISTING page: the current page number in the pagination bar (section 2).
# On screen you see "1". In the HTML there is a hidden text for screen readers:
#   <li class="active"><span>1 <span class="visually-hidden">(current)</span></span></li>
active_page = soup.select_one("ul.pagination > li.active")
print("before :", active_page.get_text(" ", strip=True))      # "1 (current)"

# remove the hidden <span> from the tree; extract() gives it back in a variable
hidden_text = active_page.select_one(".visually-hidden").extract()
print("removed:", hidden_text)

print("after  :", active_page.get_text(" ", strip=True))      # "1"

before : 1 (current)
removed: <span class="visually-hidden">(current)</span>
after  : 1


## 9. Faster? Measure first
For ten pages, parsing time does not matter (the network is 100× slower). For **thousands** of pages it does.
Two options:
- `SoupStrainer`: BeautifulSoup builds only the tags you want (less memory, a bit faster);
- **parsel** (installed: the selector library behind Scrapy): CSS and XPath directly on lxml, no BeautifulSoup tree.

Measure on YOUR page (time several runs: one run is noise).

In [93]:
from parsel import Selector

RUNS = 5    # time several runs and take the average: one run is noise

# option 1: BeautifulSoup + lxml (what we did so far)
start = time.perf_counter()
for _ in range(RUNS):
    cards_found = len(BeautifulSoup(listing_html, "lxml").select("li.pdt-item"))
ms = (time.perf_counter() - start) / RUNS * 1000
print(f"{'BeautifulSoup':30} {ms:5.0f} ms   ({cards_found} cards)")

# option 2: same, but BeautifulSoup builds ONLY the product cards
only_cards = SoupStrainer("li", class_="pdt-item")
start = time.perf_counter()
for _ in range(RUNS):
    cards_found = len(BeautifulSoup(listing_html, "lxml", parse_only=only_cards).select("li.pdt-item"))
ms = (time.perf_counter() - start) / RUNS * 1000
print(f"{'BeautifulSoup + SoupStrainer':30} {ms:5.0f} ms   ({cards_found} cards)")

# option 3: parsel, CSS selectors directly on lxml (no BeautifulSoup tree)
start = time.perf_counter()
for _ in range(RUNS):
    cards_found = len(Selector(text=listing_html).css("li.pdt-item"))
ms = (time.perf_counter() - start) / RUNS * 1000
print(f"{'parsel':30} {ms:5.0f} ms   ({cards_found} cards)")

BeautifulSoup                    129 ms   (48 cards)
BeautifulSoup + SoupStrainer      94 ms   (48 cards)
parsel                            16 ms   (48 cards)


## 10. "The price is not in the HTML" ... really?
Open the listing page in Chrome: the first card shows a price, **969€95**.
Let's look for this price in the HTML that `requests` downloaded.

In [94]:
# step 1: the price box of the first card ... is EMPTY
price_div = first.select_one(".price")
print("price box:", price_div)

# step 2: do not give up! is the price somewhere ELSE in the page? (use the price you see in Chrome)
print("is '969.95' in the page?", "969.95" in listing_html)

price box: <div class="price"></div>
is '969.95' in the page? True


**Yes, it is there.** The box is empty because a small JavaScript fills it **after** the page loads. But the price
is already in the page, inside a **JSON-LD** block: `<script type="application/ld+json">`, data the shop writes for
Google, with the name, the price and the stock of every product:

```json
{"@type": "ItemList", "itemListElement": [
  {"item": {"name": "ASRock ... 16GB OC", "offers": {"price": "969.95", "availability": "https://schema.org/InStock"}}},
  ...
]}
```

See it yourself: on the listing page press `Ctrl+U` (page source), then `Ctrl+F` → `969.95`.
It is JSON, so `json.loads` reads it: **no browser needed**.

In [95]:
# step 3: find the JSON-LD block that lists the products ("@type": "ItemList")
for block in soup.select('script[type="application/ld+json"]'):
    data = json.loads(block.string)
    if data.get("@type") == "ItemList":
        product_list = data["itemListElement"]

# step 4: read name, price and stock of the first 3 products
for element in product_list[:3]:
    item = element["item"]
    name = item["name"]
    price = item["offers"]["price"]
    availability = item["offers"]["availability"].rsplit("/", 1)[-1]   # ".../InStock" -> "InStock"
    print(price, availability, name)

969.95 InStock ASRock AMD Radeon RX 9060 XT Challenger 16GB OC
1129.95 InStock ASRock AMD Radeon RX 9070 Challenger 16GB
1329.95 InStock ASRock AMD Radeon RX 9070 XT Challenger 16GB
